# LeetCode 1680 — Concatenation of Consecutive Binary Numbers bằng Python

Link đề: https://leetcode.com/problems/concatenation-of-consecutive-binary-numbers/ · Độ khó: **Medium** · Chủ đề: Math, Bit Manipulation, Simulation

Skeleton LeetCode:

```python
class Solution:
    def concatenatedBinary(self, n: int) -> int:
        ...
```

Notebook đi từ **ghép chuỗi nhị phân thật rồi đổi sang số** → **số nguyên lớn dịch bit không lấy mod** → **dịch bit + lấy mod từng bước** (dùng `bit_length` hoặc đếm độ dài khi gặp lũy thừa của 2) → **nhóm theo độ dài bit + lũy thừa ma trận** chạy `O(log² n)`.

## 1. Mục tiêu học

1. Hiểu phép "ghép nhị phân": nối `bin(i)` vào sau ⇔ `ans = ans · 2^len(i) + i` ⇔ `ans = (ans << len) | i`.
2. Biết lấy **modulo từng bước** an toàn vì phép nhân/cộng tương thích với mod.
3. Tính độ dài bit bằng `int.bit_length()` hoặc **tăng độ dài khi `i` là lũy thừa của 2** (`i & (i − 1) == 0`) — và cẩn thận **độ ưu tiên toán tử** trong Python.
4. Nâng cao: các số cùng độ dài bit tạo thành một **phép biến đổi tuyến tính** ⇒ dùng **lũy thừa ma trận** để nhảy cả nhóm.
5. Kiểm thử bằng cách so với chuỗi nhị phân thật (`int(s, 2)`) với nhiều `n`, seed cố định.

## 2. Đề bài & ví dụ

Cho số nguyên `n`, trả về **giá trị thập phân** của xâu nhị phân tạo bằng cách **nối** biểu diễn nhị phân của `1, 2, …, n` theo thứ tự, **modulo `10⁹ + 7`**.

**Ví dụ 1:** `n = 1` → `1` (`"1"` = 1).

**Ví dụ 2:** `n = 3` → `27`. `1, 2, 3` là `"1"`, `"10"`, `"11"`; nối lại `"11011"` = 27.

**Ví dụ 3:** `n = 12` → `505379714`. Xâu nối là `"1101110010111011110001001101010111100"`, giá trị `118505380540`, mod `10⁹ + 7` được `505379714`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`.

In [1]:
import random
import time
from typing import List

MOD = 10**9 + 7
examples = [(1, 1), (3, 27), (12, 505379714)]

for n, exp in examples:
    s = ''.join(bin(i)[2:] for i in range(1, n + 1))
    print(f"n={n}: xâu = {s} -> {int(s, 2)} -> mod = {int(s, 2) % MOD} (kỳ vọng {exp})")

n=1: xâu = 1 -> 1 -> mod = 1 (kỳ vọng 1)
n=3: xâu = 11011 -> 27 -> mod = 27 (kỳ vọng 27)
n=12: xâu = 1101110010111011110001001101010111100 -> 118505380540 -> mod = 505379714 (kỳ vọng 505379714)


## 3. Kiến thức nền

### 3.1 `bin`, `int(s, 2)` và `bit_length`

`bin(6) = '0b110'` (có tiền tố `0b`); `bin(6)[2:]` bỏ tiền tố. `int('110', 2) = 6`. `(6).bit_length() = 3` là số bit **không kể số 0 đứng đầu**.

In [2]:
for x in [1, 2, 3, 6, 8, 12]:
    print(f"{x:>2}: bin={bin(x):>7}, không tiền tố={bin(x)[2:]:>5}, bit_length={x.bit_length()}")
print(int('11011', 2))

 1: bin=    0b1, không tiền tố=    1, bit_length=1
 2: bin=   0b10, không tiền tố=   10, bit_length=2
 3: bin=   0b11, không tiền tố=   11, bit_length=2
 6: bin=  0b110, không tiền tố=  110, bit_length=3
 8: bin= 0b1000, không tiền tố= 1000, bit_length=4
12: bin= 0b1100, không tiền tố= 1100, bit_length=4
27


### 3.2 Dịch bit trái `<<` và OR `|`

`a << L` = `a · 2^L` (thêm `L` số 0 vào cuối biểu diễn nhị phân). Nếu `b < 2^L` thì `(a << L) | b = a · 2^L + b` — tức là "ghép" `b` vào sau `a` (đệm cho đủ `L` bit).

In [3]:
a, b = 0b11, 0b011        # b = 3 nhưng ta muốn ghép 3 bit
L = 3
print(bin(a << L), bin((a << L) | b), (a << L) | b == a * 2**L + b)

0b11000 0b11011 True


### 3.3 Modulo tương thích với `+` và `·`

`(x · y + z) mod p = ((x mod p) · y + z) mod p`. Nhờ vậy ta lấy mod **sau mỗi bước** mà kết quả cuối không đổi, đồng thời giữ số nhỏ (nhanh). Dịch bit cũng là phép nhân với `2^L`.

In [4]:
x, L, i = 123456789123456789, 17, 99999
print(((x << L) | i) % MOD == (((x % MOD) << L) | i) % MOD == ((x % MOD) * 2**L + i) % MOD)

True


### 3.4 Lũy thừa của 2 và độ ưu tiên toán tử

`i` là lũy thừa của 2 ⇔ `i & (i − 1) == 0` (với `i > 0`): phép `i & (i − 1)` xóa bit 1 thấp nhất.

**Độ ưu tiên toán tử:** trong Python thứ tự là `+ −` > `<< >>` > `&` > `^` > `|` > so sánh (`==`, `<`…). Vì vậy:
- `i & (i - 1) == 0` trong **Python** vẫn là `(i & (i - 1)) == 0` (đúng), nhưng trong **C/C++/Java** `==` ưu tiên hơn `&` nên thành `i & ((i - 1) == 0)` — một lỗi kinh điển khi chuyển code. Thói quen tốt: **luôn thêm ngoặc**.
- `a << L + i` là `a << (L + i)` (cả Python lẫn C) — **sai** nếu ý bạn là `(a << L) + i`.

In [5]:
for i in [1, 2, 3, 4, 6, 8]:
    print(f"i={i}: (i & (i-1)) == 0 -> {(i & (i - 1)) == 0} | thiếu ngoặc trong Python -> {i & (i - 1) == 0} | kiểu C: i & ((i-1) == 0) -> {bool(i & ((i - 1) == 0))}")
print("1 << 2 + 1 =", 1 << 2 + 1, "(tức 1 << 3), còn (1 << 2) + 1 =", (1 << 2) + 1)

i=1: (i & (i-1)) == 0 -> True | thiếu ngoặc trong Python -> True | kiểu C: i & ((i-1) == 0) -> True
i=2: (i & (i-1)) == 0 -> True | thiếu ngoặc trong Python -> True | kiểu C: i & ((i-1) == 0) -> False
i=3: (i & (i-1)) == 0 -> False | thiếu ngoặc trong Python -> False | kiểu C: i & ((i-1) == 0) -> False
i=4: (i & (i-1)) == 0 -> True | thiếu ngoặc trong Python -> True | kiểu C: i & ((i-1) == 0) -> False
i=6: (i & (i-1)) == 0 -> False | thiếu ngoặc trong Python -> False | kiểu C: i & ((i-1) == 0) -> False
i=8: (i & (i-1)) == 0 -> True | thiếu ngoặc trong Python -> True | kiểu C: i & ((i-1) == 0) -> False
1 << 2 + 1 = 8 (tức 1 << 3), còn (1 << 2) + 1 = 5


## 4. Ý tưởng nền tảng / trực giác

Nối nhị phân giống viết thêm chữ số vào **cuối** một số thập phân: viết `7` sau `12` được `127 = 12 · 10 + 7`. Trong hệ 2: nối `bin(i)` (dài `L` bit) vào sau `ans` ⇔ `ans · 2^L + i`.

Tính tay `n = 4`:

| i | bin(i) | L | ans trước | ans sau = ans·2^L + i | xâu |
|---|---|---|---|---|---|
| 1 | 1 | 1 | 0 | 1 | 1 |
| 2 | 10 | 2 | 1 | 1·4 + 2 = 6 | 110 |
| 3 | 11 | 2 | 6 | 6·4 + 3 = 27 | 11011 |
| 4 | 100 | 3 | 27 | 27·8 + 4 = 220 | 11011100 |

In [6]:
ans = 0
s = ""
for i in range(1, 5):
    L = i.bit_length()
    ans = ans * 2**L + i
    s += bin(i)[2:]
    print(f"i={i}: L={L}, ans={ans}, xâu={s}, int(xâu,2)={int(s, 2)}")

i=1: L=1, ans=1, xâu=1, int(xâu,2)=1
i=2: L=2, ans=6, xâu=110, int(xâu,2)=6
i=3: L=2, ans=27, xâu=11011, int(xâu,2)=27
i=4: L=3, ans=220, xâu=11011100, int(xâu,2)=220


## 5. Cách 1 — Brute Force / baseline: ghép chuỗi thật

**Ý tưởng:** làm đúng như đề: ghép các `bin(i)[2:]`, đổi sang số bằng `int(s, 2)`, lấy mod.

In [7]:
def concat_string(n):
    s = ''.join(bin(i)[2:] for i in range(1, n + 1))
    return int(s, 2) % MOD

for n, exp in examples:
    print(concat_string(n), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
27 (kỳ vọng 27 )
505379714 (kỳ vọng 505379714 )


**Độ phức tạp:** xâu dài `Σ bit_length(i) ≈ n log₂ n` (khoảng `1.5·10⁶` ký tự với `n = 10⁵`). Ghép bằng `join` là `O(n log n)`; `int(s, 2)` với cơ số lũy thừa 2 cũng gần tuyến tính, rồi phép `%` trên số `~1.5·10⁶` bit. Bộ nhớ `O(n log n)`. Chạy được trong Python nhưng tốn bộ nhớ và **không** chuyển được sang ngôn ngữ không có số nguyên lớn.

**Nút thắt:** ta giữ toàn bộ số khổng lồ trong khi chỉ cần **phần dư**. Phép ghép là `ans · 2^L + i` — một phép toán **tương thích với mod**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (mod từng bước):** `ans ← (ans · 2^L + i) mod p` cho kết quả giống hệt việc tính số lớn rồi mới mod (mục 3.3). Mỗi bước `O(1)` ⇒ tổng `O(n)`.

**Insight 2 (độ dài bit tăng đúng lúc gặp lũy thừa của 2):** `L(i) = L(i − 1) + 1` **nếu** `i` là lũy thừa của 2, **ngược lại** `L(i) = L(i − 1)`. Không cần gọi `bit_length` mỗi lần.

**Insight 3 (nhóm cùng độ dài):** các số `i ∈ [2^(L−1), 2^L − 1]` cùng độ dài `L`. Trên nhóm đó, bước `(ans, i) → (ans·2^L + i, i + 1)` là **tuyến tính** theo vector `(ans, i, 1)`:

```
| ans' |   | 2^L  1  0 | | ans |
| i'   | = |  0   1  1 | |  i  |
| 1    |   |  0   0  1 | |  1  |
```

⇒ áp dụng `m` lần = nhân với `M^m`, tính bằng **lũy thừa nhanh** trong `O(log m)` phép nhân ma trận `3×3`. Có `~log₂ n` nhóm ⇒ `O(log² n)`.

Kiểm chứng tính tuyến tính trên nhóm `L = 3` (`i = 4..7`): áp từng bước và áp một lần `M⁴` phải cho cùng kết quả. Đồng thời liệt kê các nhóm độ dài bit với `n = 20` để thấy có rất ít nhóm (`⌊log₂ n⌋ + 1`).

In [8]:
L = 3
M = [[2**L, 1, 0], [0, 1, 1], [0, 0, 1]]
ans0 = 27                                   # giá trị sau khi ghép 1..3
v = [ans0, 4, 1]
for _ in range(4):                          # áp từng bước
    v = [M[0][0] * v[0] + M[0][1] * v[1] + M[0][2] * v[2], v[1] + 1, 1]
step_by_step = v[0]
M2 = [[sum(M[i][k] * M[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
M4 = [[sum(M2[i][k] * M2[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
by_matrix = M4[0][0] * ans0 + M4[0][1] * 4 + M4[0][2] * 1
print("từng bước:", step_by_step, "| M^4 một lần:", by_matrix, "| xâu thật:", int(''.join(bin(i)[2:] for i in range(1, 8)), 2))

n = 20
groups = {}
for i in range(1, n + 1):
    groups.setdefault(i.bit_length(), []).append(i)
for L_, members in groups.items():
    print(f"L={L_}: {members}")

từng bước: 113015 | M^4 một lần: 113015 | xâu thật: 113015
L=1: [1]
L=2: [2, 3]
L=3: [4, 5, 6, 7]
L=4: [8, 9, 10, 11, 12, 13, 14, 15]
L=5: [16, 17, 18, 19, 20]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Số nguyên lớn, dịch bit, **không** lấy mod giữa chừng

In [9]:
def concat_bigint(n):
    ans = 0
    for i in range(1, n + 1):
        ans = (ans << i.bit_length()) | i
    return ans % MOD

for n, exp in examples:
    print(concat_bigint(n), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
27 (kỳ vọng 27 )
505379714 (kỳ vọng 505379714 )


Đúng, nhưng `ans` lớn dần tới `~n log n` bit; mỗi lần dịch phải **sao chép** cả số ⇒ tổng `O((n log n)² / w)` với `w` là độ dài từ máy — chậm rõ rệt khi `n` lớn (mục 11).

### 7.2 Cách 3 — Dịch bit + mod từng bước với `bit_length`

In [10]:
def concat_mod_bitlength(n):
    ans = 0
    for i in range(1, n + 1):
        ans = ((ans << i.bit_length()) | i) % MOD
    return ans

for n, exp in examples:
    print(concat_mod_bitlength(n), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
27 (kỳ vọng 27 )
505379714 (kỳ vọng 505379714 )


Chú ý `|` đúng bằng `+` ở đây vì `ans << L` có `L` bit cuối bằng 0 và `i < 2^L` (trước khi mod). Sau khi mod, `ans` có thể có bit ở mọi vị trí, nhưng ta luôn dịch **trước** rồi mới OR, nên vẫn đúng. **Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Đếm độ dài bằng lũy thừa của 2

In [11]:
def concat_mod_shift(n):
    ans = 0
    L = 0
    for i in range(1, n + 1):
        if (i & (i - 1)) == 0:           # i là lũy thừa của 2 -> độ dài tăng 1
            L += 1
        ans = ((ans << L) | i) % MOD
    return ans

class SolutionShift:
    def concatenatedBinary(self, n: int) -> int:
        MOD = 10**9 + 7
        ans = L = 0
        for i in range(1, n + 1):
            if (i & (i - 1)) == 0:
                L += 1
            ans = ((ans << L) | i) % MOD
        return ans

for n, exp in examples:
    print(concat_mod_shift(n), SolutionShift().concatenatedBinary(n), "(kỳ vọng", exp, ")")

1 1 (kỳ vọng 1 )
27 27 (kỳ vọng 27 )
505379714 505379714 (kỳ vọng 505379714 )


Nhớ **ngoặc** quanh `i & (i - 1)` (mục 3.4) — bắt buộc nếu chuyển sang C/C++/Java. **Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.4 Cách 5 — Nhóm theo độ dài bit + lũy thừa ma trận `O(log² n)`

In [12]:
def mat_mul(A, B, mod=MOD):
    return [[sum(A[i][k] * B[k][j] for k in range(3)) % mod for j in range(3)] for i in range(3)]

def mat_pow(M, e, mod=MOD):
    R = [[int(i == j) for j in range(3)] for i in range(3)]
    while e:
        if e & 1:
            R = mat_mul(R, M, mod)
        M = mat_mul(M, M, mod)
        e >>= 1
    return R

def concat_matrix(n):
    ans = 0
    L = 1
    while (1 << (L - 1)) <= n:
        lo = 1 << (L - 1)                      # số nhỏ nhất có L bit
        hi = min(n, (1 << L) - 1)              # số lớn nhất có L bit (không vượt n)
        M = [[pow(2, L, MOD), 1, 0], [0, 1, 1], [0, 0, 1]]
        P = mat_pow(M, hi - lo + 1)
        v = (ans, lo % MOD, 1)
        ans = (P[0][0] * v[0] + P[0][1] * v[1] + P[0][2] * v[2]) % MOD
        L += 1
    return ans

for n, exp in examples:
    print(concat_matrix(n), "(kỳ vọng", exp, ")")
print("n = 10^18 (chỉ cách ma trận làm được):", concat_matrix(10**18))

1 (kỳ vọng 1 )
27 (kỳ vọng 27 )
505379714 (kỳ vọng 505379714 )
n = 10^18 (chỉ cách ma trận làm được): 945204784


**Độ phức tạp:** có `⌊log₂ n⌋ + 1` nhóm; mỗi nhóm lũy thừa ma trận `O(log n)` phép nhân `3×3` ⇒ `O(log² n)`. Với ràng buộc `n ≤ 10⁵` thì cách 3/4 đã đủ, nhưng cách này cho phép `n` tới `10¹⁸`.

## 8. TRACE từng bước

Trace cách 4 trên `n = 12` (in mỗi bước: `i`, có phải lũy thừa 2, `L`, `ans` mod, và kiểm với xâu thật):

| i | lũy thừa 2? | L |
|---|---|---|
| 1 | ✔ | 1 |
| 2 | ✔ | 2 |
| 3 | | 2 |
| 4 | ✔ | 3 |
| 5–7 | | 3 |
| 8 | ✔ | 4 |
| 9–12 | | 4 |

In [13]:
def trace_shift(n):
    ans = L = 0
    s = ""
    for i in range(1, n + 1):
        p2 = (i & (i - 1)) == 0
        if p2:
            L += 1
        ans = ((ans << L) | i) % MOD
        s += bin(i)[2:]
        print(f"i={i:>2} bin={bin(i)[2:]:>4} lũy thừa 2={'✔' if p2 else ' '} L={L} ans={ans:>10} | int(xâu) % MOD = {int(s, 2) % MOD:>10}")
    return ans

trace_shift(12)

i= 1 bin=   1 lũy thừa 2=✔ L=1 ans=         1 | int(xâu) % MOD =          1
i= 2 bin=  10 lũy thừa 2=✔ L=2 ans=         6 | int(xâu) % MOD =          6
i= 3 bin=  11 lũy thừa 2=  L=2 ans=        27 | int(xâu) % MOD =         27
i= 4 bin= 100 lũy thừa 2=✔ L=3 ans=       220 | int(xâu) % MOD =        220
i= 5 bin= 101 lũy thừa 2=  L=3 ans=      1765 | int(xâu) % MOD =       1765
i= 6 bin= 110 lũy thừa 2=  L=3 ans=     14126 | int(xâu) % MOD =      14126
i= 7 bin= 111 lũy thừa 2=  L=3 ans=    113015 | int(xâu) % MOD =     113015
i= 8 bin=1000 lũy thừa 2=✔ L=4 ans=   1808248 | int(xâu) % MOD =    1808248
i= 9 bin=1001 lũy thừa 2=  L=4 ans=  28931977 | int(xâu) % MOD =   28931977
i=10 bin=1010 lũy thừa 2=  L=4 ans= 462911642 | int(xâu) % MOD =  462911642
i=11 bin=1011 lũy thừa 2=  L=4 ans= 406586234 | int(xâu) % MOD =  406586234
i=12 bin=1100 lũy thừa 2=  L=4 ans= 505379714 | int(xâu) % MOD =  505379714


505379714

Trace cách 5 (ma trận) trên `n = 12`: bốn nhóm `L = 1: [1, 1]`, `L = 2: [2, 3]`, `L = 3: [4, 7]`, `L = 4: [8, 12]`.

In [14]:
def trace_matrix(n):
    ans, L = 0, 1
    while (1 << (L - 1)) <= n:
        lo, hi = 1 << (L - 1), min(n, (1 << L) - 1)
        M = [[pow(2, L, MOD), 1, 0], [0, 1, 1], [0, 0, 1]]
        P = mat_pow(M, hi - lo + 1)
        new = (P[0][0] * ans + P[0][1] * lo + P[0][2]) % MOD
        print(f"nhóm L={L}: i ∈ [{lo}, {hi}] ({hi - lo + 1} số), M^{hi - lo + 1}[0] = {P[0]}, ans: {ans} -> {new}")
        ans = new; L += 1
    return ans

print("kết quả:", trace_matrix(12))

nhóm L=1: i ∈ [1, 1] (1 số), M^1[0] = [2, 1, 0], ans: 0 -> 1
nhóm L=2: i ∈ [2, 3] (2 số), M^2[0] = [16, 5, 1], ans: 1 -> 27
nhóm L=3: i ∈ [4, 7] (4 số), M^4[0] = [4096, 585, 83], ans: 27 -> 113015
nhóm L=4: i ∈ [8, 12] (5 số), M^5[0] = [1048576, 69905, 4660], ans: 113015 -> 505379714
kết quả: 505379714


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def concatenatedBinary(self, n: int) -> int:
        MOD = 10**9 + 7
        ans = 0
        L = 0                                   # độ dài bit của i hiện tại
        for i in range(1, n + 1):
            if (i & (i - 1)) == 0:              # i là lũy thừa của 2
                L += 1
            ans = ((ans << L) | i) % MOD
        return ans

for n, exp in examples:
    assert Solution().concatenatedBinary(n) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `X_i` là số nguyên ứng với xâu `bin(1) bin(2) … bin(i)`. Theo định nghĩa phép nối nhị phân: `X_i = X_{i−1} · 2^{L(i)} + i` với `L(i) = bit_length(i)`, `X_0 = 0`. (Nối `L` bit của `i` vào cuối = dịch trái `L` rồi đặt `i` vào `L` bit thấp; vì `i < 2^{L(i)}` nên phép đặt là phép cộng.)

**Mod từng bước:** bằng quy nạp, `ans_i ≡ X_i (mod p)`: `ans_i = (ans_{i−1} · 2^{L} + i) mod p ≡ X_{i−1} · 2^L + i = X_i`. Trong code, `(ans << L) | i`: `ans << L` có `L` bit cuối bằng 0 và `i < 2^L` ⇒ `|` = `+`.

**Độ dài bit:** `L(i) = ⌊log₂ i⌋ + 1` tăng đúng 1 tại `i = 2^t` và giữ nguyên giữa hai lũy thừa liên tiếp. Với `i > 0`, `i & (i − 1)` xóa bit 1 thấp nhất; bằng 0 ⇔ `i` chỉ có một bit 1 ⇔ `i` là lũy thừa của 2.

**Ma trận:** trên nhóm `L` cố định, bước `(X, i, 1) ↦ (2^L X + i, i + 1, 1)` là phép nhân với `M`; áp `m` bước liên tiếp = nhân với `M^m` (tính kết hợp của phép nhân ma trận), mọi phép tính lấy mod `p` vẫn đúng vì là tổ hợp cộng/nhân.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Ghép chuỗi + `int(s, 2)` | `O(n log n)` | `O(n log n)` |
| 2. Số lớn, không mod | `O((n log n)² / w)` | `O(n log n)` bit |
| 3. Mod từng bước (`bit_length`) | `O(n)` | `O(1)` |
| 4. Mod từng bước (lũy thừa 2) | `O(n)` | `O(1)` |
| 5. Ma trận theo nhóm | `O(log² n)` | `O(1)` |

In [16]:
N = 10**5
for name, fn in [("string", concat_string), ("mod_bitlength", concat_mod_bitlength), ("mod_shift", concat_mod_shift), ("matrix", concat_matrix), ("Solution", Solution().concatenatedBinary)]:
    t0 = time.perf_counter(); r = fn(N); dt = time.perf_counter() - t0
    print(f"{name:<14} n=10^5: {r:>10}  {dt * 1000:8.1f} ms")
for m in (5000, 20000):
    t0 = time.perf_counter(); r = concat_bigint(m); dt = time.perf_counter() - t0
    print(f"bigint (không mod) n={m}: {dt * 1000:.1f} ms | khớp: {r == concat_mod_shift(m)}")

string         n=10^5:  757631812      15.9 ms
mod_bitlength  n=10^5:  757631812       6.5 ms
mod_shift      n=10^5:  757631812       8.3 ms
matrix         n=10^5:  757631812       0.6 ms
Solution       n=10^5:  757631812       8.3 ms
bigint (không mod) n=5000: 3.1 ms | khớp: True
bigint (không mod) n=20000: 52.2 ms | khớp: True


Bản "số lớn không mod" tăng thời gian nhanh hơn tuyến tính khi `n` tăng (dấu hiệu bậc hai) — với `n = 10⁵` sẽ chậm hơn hẳn các bản lấy mod.

## 12. Bộ test

In [17]:
ALL = {
    "string": concat_string,
    "mod_bitlength": concat_mod_bitlength,
    "mod_shift": concat_mod_shift,
    "matrix": concat_matrix,
    "SolShift": lambda n: SolutionShift().concatenatedBinary(n),
    "Solution": lambda n: Solution().concatenatedBinary(n),
}

def check(n, exp=None):
    ref = concat_string(n)
    for name, fn in ALL.items():
        got = fn(n)
        assert got == ref, (name, n, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for n, exp in examples:
    check(n, exp)
print("Ví dụ đề: OK")

for n in list(range(1, 300)) + [511, 512, 513, 1023, 1024, 1025, 65535, 65536, 65537, 99999, 100000]:
    check(n)
print("n = 1..299 và các biên quanh lũy thừa của 2 (tới 10^5): OK")

rnd = random.Random(1680)
for _ in range(30):
    check(rnd.randint(1, 10**5))
print("30 giá trị n ngẫu nhiên: OK")

# kiểm ma trận với n lớn hơn ràng buộc bằng cách so với bản mod từng bước ở n vừa phải
for n in [123456, 200000, 262144, 262145]:
    assert concat_matrix(n) == concat_mod_shift(n)
print("Ma trận khớp với mod từng bước cho n tới 2.6·10^5: OK")

Ví dụ đề: OK


n = 1..299 và các biên quanh lũy thừa của 2 (tới 10^5): OK


30 giá trị n ngẫu nhiên: OK
Ma trận khớp với mod từng bước cho n tới 2.6·10^5: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dịch 1 bit thay vì `bit_length(i)` bit

In [18]:
def concat_bug_shift1(n):
    ans = 0
    for i in range(1, n + 1):
        ans = (ans * 2 + i) % MOD                  # SAI: chỉ "nhân 2" như thêm 1 bit
    return ans

print("n=3: sai =", concat_bug_shift1(3), "| đúng =", Solution().concatenatedBinary(3))

n=3: sai = 11 | đúng = 27


**Vì sao sai:** nối `bin(i)` dài `L` bit cần dịch `L` vị trí (`· 2^L`), không phải 1. **Sửa:** `ans << i.bit_length()`.

### Lỗi 2 — Độ ưu tiên toán tử: `ans << L + i`

In [19]:
def concat_bug_precedence(n):
    ans = 0
    for i in range(1, n + 1):
        ans = (ans << i.bit_length() + i) % MOD    # SAI: hiểu là ans << (L + i)
    return ans

print("n=3: sai =", concat_bug_precedence(3), "| đúng =", Solution().concatenatedBinary(3))

n=3: sai = 0 | đúng = 27


**Vì sao sai:** trong Python, `+` ưu tiên hơn `<<` ⇒ `ans << (L + i)`, và `i` không hề được cộng vào. **Sửa:** `((ans << L) | i)` hoặc `(ans << L) + i` có ngoặc.

### Lỗi 3 — Tăng độ dài `L` **sau** khi đã ghép `i`

In [20]:
def concat_bug_late_L(n):
    ans, L = 0, 0
    for i in range(1, n + 1):
        ans = ((ans << L) | i) % MOD             # SAI: dùng L cũ cho i là lũy thừa của 2
        if (i & (i - 1)) == 0:
            L += 1
    return ans

def concat_bug_c_precedence(n):
    # Mô phỏng code C/Java viết `if (i & (i - 1) == 0)`: thực chất là i & ((i-1) == 0)
    ans, L = 0, 0
    for i in range(1, n + 1):
        if i & ((i - 1) == 0):
            L += 1
        ans = ((ans << L) | i) % MOD
    return ans

for n in [3, 4, 12]:
    print(f"n={n}: tăng L muộn = {concat_bug_late_L(n)}, kiểu C thiếu ngoặc = {concat_bug_c_precedence(n)}, đúng = {Solution().concatenatedBinary(n)}")

n=3: tăng L muộn = 11, kiểu C thiếu ngoặc = 7, đúng = 27
n=4: tăng L muộn = 44, kiểu C thiếu ngoặc = 14, đúng = 220
n=12: tăng L muộn = 7807592, kiểu C thiếu ngoặc = 4094, đúng = 505379714


**Vì sao sai:** với `i = 2` (vừa sang độ dài 2 bit) bản "tăng muộn" vẫn dịch 1 bit ⇒ bit của `i` chồng lên `ans` (và với `i = 1` còn dịch 0 bit). Bản "kiểu C" chỉ tăng `L` tại `i = 1`, nên `L` kẹt ở 1. **Sửa:** cập nhật `L` **trước** khi ghép, và luôn viết `(i & (i - 1)) == 0`.

### Lỗi 4 — Dùng `bin(i)` kèm tiền tố `'0b'` khi ghép chuỗi

In [21]:
def concat_bug_prefix(n):
    s = ''.join(bin(i) for i in range(1, n + 1))   # SAI: có '0b' ở giữa xâu
    try:
        return int(s, 2) % MOD
    except ValueError as e:
        return f"ValueError: {e}"

print("n=3:", concat_bug_prefix(3), "| đúng =", Solution().concatenatedBinary(3))

n=3: ValueError: invalid literal for int() with base 2: '0b10b100b11' | đúng = 27


**Sửa:** `bin(i)[2:]` hoặc `format(i, 'b')`.

### Lỗi 5 — Lấy mod nhưng **cộng** trước khi dịch: `((ans | i) << L) % MOD`

In [22]:
def concat_bug_order(n):
    ans = 0
    for i in range(1, n + 1):
        ans = ((ans | i) << i.bit_length()) % MOD   # SAI: dịch sau khi chèn i
    return ans

print("n=3: sai =", concat_bug_order(3), "| đúng =", Solution().concatenatedBinary(3))

n=3: sai = 44 | đúng = 27


**Vì sao sai:** phải **dời** phần cũ sang trái để chừa chỗ, **rồi** đặt `i` vào các bit thấp. Làm ngược sẽ chồng `i` lên bit của `ans` và thêm các số 0 thừa ở cuối. **Sửa:** `((ans << L) | i)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nối chữ số/bit rồi lấy giá trị mod `p`" ⇒ `ans = (ans · base^len + x) mod p` — giống đọc số từng chữ số (Horner).
- `bit_length()` cho độ dài nhị phân; hoặc tăng độ dài khi gặp lũy thừa 2.
- Cẩn thận độ ưu tiên: trong Python `+ −` > `<< >>` > `&` > `^` > `|` > so sánh; trong C/C++/Java so sánh lại **cao hơn** `&`, `|`. **Luôn thêm ngoặc**.
- Bước cập nhật tuyến tính và lặp với cùng tham số nhiều lần ⇒ **lũy thừa ma trận** nhảy cả đoạn.
- Python có số nguyên lớn — tiện để **kiểm thử** (so với `int(s, 2)`), nhưng đừng dựa vào nó cho lời giải chính.

## 15. Biến thể của bài

1. **Nối thập phân** `"123456789101112…n"` mod `p`: `ans = ans · 10^digits(i) + i`.
2. **Nối theo thứ tự ngược** (`n, n−1, …, 1`): thêm vào **đầu** ⇒ cần biết tổng độ dài phía sau: `ans += i · 2^{tổng độ dài các số sau}`.
3. **Nối các số trong mảng bất kỳ:** cùng công thức Horner với độ dài từng phần tử.
4. **`n` rất lớn (tới `10¹⁸`):** cách ma trận theo nhóm độ dài.
5. **LeetCode 1009 / 476 — Complement of Base 10 Integer:** thao tác bit với `bit_length`.
6. **Đếm số bit 1 trong xâu nối:** `Σ popcount(i)` — có công thức theo từng bit.

In [23]:
def concat_decimal(n, mod=MOD):
    ans = 0
    for i in range(1, n + 1):
        ans = (ans * pow(10, len(str(i)), mod) + i) % mod
    return ans

def concat_reverse_binary(n, mod=MOD):
    """Biến thể 2: xâu bin(n) bin(n-1) ... bin(1)."""
    ans, shift = 0, 0
    for i in range(1, n + 1):                      # đi từ phần cuối xâu lên đầu
        ans = (ans + i * pow(2, shift, mod)) % mod
        shift += i.bit_length()
    return ans

print("thập phân n=12:", concat_decimal(12), "| kiểm:", int(''.join(map(str, range(1, 13)))) % MOD)
print("nhị phân ngược n=5:", concat_reverse_binary(5), "| kiểm:", int(''.join(bin(i)[2:] for i in range(5, 0, -1)), 2) % MOD)

thập phân n=12: 788236920 | kiểm: 788236920
nhị phân ngược n=5: 1437 | kiểm: 1437


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép chuỗi | đúng theo định nghĩa | `O(n log n)` | `O(n log n)` | kiểm thử |
| 2. Số lớn | dịch bit, mod cuối | `O((n log n)²/w)` | lớn | minh họa vì sao cần mod |
| 3. Mod + `bit_length` | Horner nhị phân | `O(n)` | `O(1)` | dễ viết |
| 4. Mod + lũy thừa 2 | tăng `L` khi `i = 2^t` | `O(n)` | `O(1)` | **nộp bài** (nhanh, không gọi hàm) |
| 5. Ma trận | nhóm cùng độ dài, `M^m` | `O(log² n)` | `O(1)` | `n` cực lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().concatenatedBinary(2)` và `Solution().concatenatedBinary(4)`?

**Bài 2 (tính tay):** Tính tay giá trị (chưa mod) của xâu nối với `n = 5` bằng công thức `ans = ans · 2^L + i`, lập bảng như mục 4.

**Bài 3 (điền TODO):** Hoàn thiện `concat_array(nums)` trả về giá trị (mod `10⁹ + 7`) của xâu nối `bin(nums[0]) bin(nums[1]) …` cho mảng số dương bất kỳ.

**Bài 4 (tự viết lại):** Viết lại cách 3 **không dùng** `bit_length` và **không dùng** `&`: duy trì `next_pow = 1` và tăng `L` khi `i == next_pow` (rồi `next_pow *= 2`).

**Bài 5 (mini challenge):** Viết `concat_mod_any(n, p)` dùng cách ma trận cho **modulo bất kỳ** `p` và `n ≤ 10¹⁸`; kiểm với cách 4 (cho `n` vừa phải) với nhiều `p`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def concat_array(nums):
    # TODO: ans = ((ans << x.bit_length()) | x) % MOD cho mỗi x
    pass

# Bài 4 — TODO
def concat_next_pow(n):
    # TODO: next_pow = 1; L = 0; nếu i == next_pow: L += 1, next_pow *= 2
    pass

# Bài 5 — TODO
def concat_mod_any(n, p):
    # TODO: như concat_matrix nhưng mọi phép % dùng p
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `n = 2`: xâu `"110"`.
- Mức 2: `n = 4`: xâu `"11011100"`.
- Mức 3: `6` và `220`.

**Bài 2**
- Mức 1: Tiếp bảng mục 4 thêm dòng `i = 5`.
- Mức 2: `bin(5) = 101`, `L = 3`.
- Mức 3: `220 · 8 + 5`.

**Bài 3**
- Mức 1: Giống hệt cách 3 nhưng duyệt mảng.
- Mức 2: `x.bit_length()` với `x > 0`.
- Mức 3: Kiểm bằng `int(''.join(bin(x)[2:] for x in nums), 2) % MOD`.

**Bài 4**
- Mức 1: Lũy thừa 2 tiếp theo cần chờ.
- Mức 2: Khi `i` chạm `next_pow` thì độ dài tăng.
- Mức 3: So với `Solution` cho `n = 1..2000`.

**Bài 5**
- Mức 1: `pow(2, L, p)` trong ma trận.
- Mức 2: Số nhóm `≈ 60` khi `n ≤ 10¹⁸`.
- Mức 3: Cẩn thận `p = 1` (mọi thứ là 0).

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().concatenatedBinary(2), Solution().concatenatedBinary(4))
print("Bài 1:", r1)
assert r1 == (6, 220)

# Bài 2
ans = 0
for i in range(1, 6):
    L = i.bit_length(); ans = ans * 2**L + i
    print(f"  i={i}: L={L}, ans={ans}")
assert ans == int(''.join(bin(i)[2:] for i in range(1, 6)), 2) == 1765

# Bài 3
def concat_array_sol(nums):
    ans = 0
    for x in nums:
        ans = ((ans << x.bit_length()) | x) % MOD
    return ans

rnd = random.Random(3)
for _ in range(500):
    nums = [rnd.randint(1, 10**9) for _ in range(rnd.randint(1, 50))]
    assert concat_array_sol(nums) == int(''.join(bin(x)[2:] for x in nums), 2) % MOD
print("Bài 3 OK | [5, 1, 2] ->", concat_array_sol([5, 1, 2]))

Bài 1: (6, 220)
  i=1: L=1, ans=1
  i=2: L=2, ans=6
  i=3: L=2, ans=27
  i=4: L=3, ans=220
  i=5: L=3, ans=1765
Bài 3 OK | [5, 1, 2] -> 46


In [26]:
# Bài 4
def concat_next_pow_sol(n):
    ans, L, next_pow = 0, 0, 1
    for i in range(1, n + 1):
        if i == next_pow:
            L += 1
            next_pow *= 2
        ans = ((ans << L) | i) % MOD
    return ans

assert all(concat_next_pow_sol(n) == Solution().concatenatedBinary(n) for n in range(1, 2001))
print("Bài 4 OK")

# Bài 5
def concat_mod_any_sol(n, p):
    def mul(A, B):
        return [[sum(A[i][k] * B[k][j] for k in range(3)) % p for j in range(3)] for i in range(3)]
    def mpow(M, e):
        R = [[int(i == j) % p for j in range(3)] for i in range(3)]
        while e:
            if e & 1: R = mul(R, M)
            M = mul(M, M); e >>= 1
        return R
    ans, L = 0, 1
    while (1 << (L - 1)) <= n:
        lo, hi = 1 << (L - 1), min(n, (1 << L) - 1)
        P = mpow([[pow(2, L, p), 1, 0], [0, 1, 1], [0, 0, 1]], hi - lo + 1)
        ans = (P[0][0] * ans + P[0][1] * lo + P[0][2]) % p
        L += 1
    return ans

def concat_mod_p_linear(n, p):
    ans = 0
    for i in range(1, n + 1):
        ans = ((ans << i.bit_length()) | i) % p
    return ans

rnd = random.Random(5)
for _ in range(60):
    n = rnd.randint(1, 3000); p = rnd.randint(1, 10**9)
    assert concat_mod_any_sol(n, p) == concat_mod_p_linear(n, p), (n, p)
assert concat_mod_any_sol(12, MOD) == 505379714
print("Bài 5 OK | n=10^18, p=998244353 ->", concat_mod_any_sol(10**18, 998244353))

Bài 4 OK
Bài 5 OK | n=10^18, p=998244353 -> 156495389


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "nối rồi lấy mod" (Horner):**

```python
ans, L = 0, 0
for i in range(1, n + 1):
    if (i & (i - 1)) == 0:      # NGOẶC ĐẦY ĐỦ
        L += 1
    ans = ((ans << L) | i) % MOD
return ans
```

**Template lũy thừa ma trận 3×3:** `M = [[2^L, 1, 0], [0, 1, 1], [0, 0, 1]]`, áp `M^(số phần tử nhóm)` lên `(ans, lo, 1)`.

**Checklist:**
- [ ] Dịch đúng `bit_length(i)` bit (không phải 1)?
- [ ] Ngoặc cho `(ans << L) | i` và `(i & (i − 1)) == 0`?
- [ ] Dịch **trước**, đặt `i` **sau**?
- [ ] Lấy mod mỗi bước?
- [ ] `bin(i)[2:]` nếu ghép chuỗi để kiểm thử?
- [ ] Đã so với `int(xâu, 2) % MOD` cho nhiều `n`, kể cả quanh lũy thừa của 2?
- [ ] Chữ ký đúng: `def concatenatedBinary(self, n: int) -> int`?